# 03 - Regridding the catalogue onto a uniform lat/lon grid

The catalogue produced by notebook 02 describes each observation as an irregular
quadrilateral: four corner coordinates, no alignment between consecutive rows, and heavy
overlap wherever orbital passes cross. That geometry is awkward for both of the renderers
in this repository, because neither of them wants to intersect polygons per frame.

This notebook converts it into something rectangular. The observation extent is covered
with a regular 0.1 deg grid, a KD-tree answers "which observation is nearest?" for every
grid node in a single vectorised query, and each node inherits that observation's element
ratios. Where several catalogue rows land on the same node - the redundancy that
overlapping passes create - their values are averaged rather than discarded, which is the
only place in the pipeline where independent measurements of the same patch of surface are
actually combined.

**Expected input:** the concatenated catalogue CSV from notebook 02, carrying
`V0_lat`..`V3_lon`, `<El>_area` and `<El>/Si_uncertainty` columns.

**Expected output:** a grid CSV with one row per 0.1 deg cell and one
`"<value> ± <uncertainty>"` string per element ratio, ready for notebook 04.

**Runtime note:** the grid is several million rows wide. Expect this notebook to want
16 GB of RAM or more; on a smaller machine, process the catalogue in slices and use the
concatenation step further down.

## Locating the input catalogue

Point `CATALOGUE_CSV` at the concatenated output of notebook 02. The `gdown` line below is
the alternative used during development, where the catalogue was kept in cloud storage
rather than in the repository - it is left in place because the file is far too large to
version here. See `data/README.md`.

In [ ]:
import os

# Local path to the catalogue produced by notebook 02.
CATALOGUE_CSV = os.environ.get("CATALOGUE_CSV", "../data/interim/coadded_catalogue.csv")

# Grid spacing in degrees. 0.1 deg is finer than the ~12.5 km native footprint, which is
# what allows overlapping passes to resolve detail below the footprint scale.
GRID_STEP = 0.1

# Where the regridded table is written.
GRID_CSV = os.environ.get("GRID_CSV", "../data/interim/grid_ratios.csv")

In [ ]:
# Development fallback: fetch the catalogue from cloud storage when it is not on disk.
# Replace the identifier with your own copy if you re-host the file.
# !gdown 1rcqUD8BJiDP8hyfJ9cgKl2g4dC5NErtf

## Nearest-neighbour regridding

`cKDTree` is built over the first corner of every footprint, and queried once for the whole
grid. The query runs in compiled code across all nodes at once; the equivalent Python loop
over the same number of nodes takes minutes rather than the second or so this needs.

Ratios are formed against silicon at this point, and the uncertainties computed in notebook
02 are carried across unchanged.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.spatial import cKDTree

df = pd.read_csv(CATALOGUE_CSV)
# Bounding box of the surveyed region, taken over all four footprint corners.
lat_min=df[["V0_lat", "V1_lat", "V2_lat", "V3_lat"]].min().min()
lat_max=df[["V0_lat", "V1_lat", "V2_lat", "V3_lat"]].max().max()
lon_min=df[["V0_lon", "V1_lon", "V2_lon", "V3_lon"]].min().min()
lon_max=df[["V0_lon", "V1_lon", "V2_lon", "V3_lon"]].max().max()

# Regular grid spanning that box at GRID_STEP resolution.
latitudes=np.arange(lat_min,lat_max,GRID_STEP)
longitudes=np.arange(lon_min,lon_max,GRID_STEP)
grid_points=np.array(np.meshgrid(latitudes,longitudes)).T.reshape(-1,2)

# Spatial index over the catalogue footprints.
grid_df=pd.DataFrame(grid_points,columns=["latitude","longitude"])
tree=cKDTree(df[["V0_lat","V0_lon"]])
distances,indices=tree.query(grid_df[["latitude","longitude"]],k=1)

# One batched query resolves every grid node at once.
distances, indices = tree.query(grid_points)

# Each node inherits the element ratios of its nearest observation.
grid_df = pd.DataFrame(grid_points, columns=["latitude", "longitude"])
grid_df["Si_area"] = df.iloc[indices]["Si_area"].values
grid_df["O/Si_ratio"] = df.iloc[indices]["O_area"].values / df.iloc[indices]["Si_area"].values
grid_df["Na/Si_ratio"] = df.iloc[indices]["Na_area"].values/ df.iloc[indices]["Si_area"].values
grid_df["Mg/Si_ratio"] = df.iloc[indices]["Al_area"].values/ df.iloc[indices]["Si_area"].values
grid_df["Al/Si_ratio"] = df.iloc[indices]["Mg_area"].values/ df.iloc[indices]["Si_area"].values
grid_df["Ca/Si_ratio"] = df.iloc[indices]["Ca_area"].values/ df.iloc[indices]["Si_area"].values
grid_df["Ti/Si_ratio"] = df.iloc[indices]["Ti_area"].values/ df.iloc[indices]["Si_area"].values
grid_df["Mn/Si_ratio"] = df.iloc[indices]["Mn_area"].values/ df.iloc[indices]["Si_area"].values
grid_df["Fe/Si_ratio"] = df.iloc[indices]["Fe_area"].values/ df.iloc[indices]["Si_area"].values
grid_df["O/Si_uncertainity"] = df.iloc[indices]["O/Si_uncertainty"].values
grid_df["Na/Si_uncertainity"] = df.iloc[indices]["Na/Si_uncertainty"].values
grid_df["Mg/Si_uncertainity"] = df.iloc[indices]["Mg/Si_uncertainty"].values
grid_df["Al/Si_uncertainity"] = df.iloc[indices]["Al/Si_uncertainty"].values
grid_df["Ca/Si_uncertainity"] = df.iloc[indices]["Ca/Si_uncertainty"].values
grid_df["Ti/Si_uncertainity"] = df.iloc[indices]["Ti/Si_uncertainty"].values
grid_df["Mn/Si_uncertainity"] = df.iloc[indices]["Mn/Si_uncertainty"].values
grid_df["Fe/Si_uncertainity"] = df.iloc[indices]["Fe/Si_uncertainty"].values

## Discarding unobserved nodes and filling gaps

Grid nodes whose nearest neighbour has no usable silicon fit cannot be normalised at all,
so they are dropped outright. Remaining gaps are filled conservatively: a missing ratio
becomes zero, while a missing uncertainty becomes a small positive floor rather than zero,
so that downstream formatting never claims a measurement was exact.

In [ ]:
pd.set_option('future.no_silent_downcasting',True)

# Keep only nodes backed by a usable silicon measurement.
new_df=grid_df[grid_df["Si_area"].notna()]
new_df.head()

In [ ]:
# Column groups. Note the 'uncertainity' spelling is carried over from the
# catalogue schema and is matched literally here.
ratio_columns=[col for col in new_df.columns if 'ratio' in col]
uncertainty_columns=[col for col in new_df.columns if 'uncertainity' in col]
new_df[uncertainty_columns]=new_df[uncertainty_columns].fillna(0.00001)

# Missing ratios become zero; missing uncertainties keep the floor set above.
other_columns=[col for col in new_df.columns if col not in uncertainty_columns+['latitude', 'longitude']]
new_df[other_columns]= new_df[other_columns].fillna(0)
new_df.head()

## Packing value and uncertainty into one field

The viewer in notebook 04 shows a ratio and its uncertainty side by side. Rather than
carrying sixteen columns through to the renderer, each pair is collapsed into a single
`"value ± uncertainty"` string here, halving the width of the table that eventually has to
be uploaded into GPU memory.

In [ ]:
# Collapse each (ratio, uncertainty) pair into one display-ready string.
combined_ratios = {}
for ratio, uncertainty in zip(ratio_columns, uncertainty_columns):
    combined_ratios[ratio] = new_df.apply(
        lambda row: f"{row[ratio]:.5f} ± {row[uncertainty]:.5f}"
        if pd.notnull(row[ratio]) and pd.notnull(row[uncertainty])
        else None,
        axis=1
    )

# Rebuild the table around the coordinate columns.
new_df1 = new_df[['latitude', 'longitude']].copy()

# Attach the formatted columns.
for key, value in combined_ratios.items():
    new_df1[key] = value.values
new_df1.head()

## Optional: rejoining a catalogue that was processed in slices

The full grid does not fit in memory on a modest machine, so during development the
catalogue was split, regridded slice by slice, and the partial grids concatenated here.
Skip this section if the previous cells ran over the whole catalogue in one pass.

In [ ]:
# Partial grid files written by earlier passes, if the catalogue was processed in slices.
# Point this at the directory holding them.
# !gdown <file-id>   # fetch each partial grid from cloud storage if not held locally

In [ ]:
import glob

# Concatenate every partial grid into one table.

path=os.path.join(os.path.dirname(GRID_CSV), "grid_part_*.csv")
csv_files=glob.glob(path)
df4_list=[]
for file in csv_files:
  df4=pd.read_csv(file)
  df4_list.append(df4)
df3=pd.concat(df4_list,ignore_index=True)
# df3 now holds the union of the partial grids; feed it into the averaging step below
# by assigning `new_df1 = df3` before running the next cell.

## Averaging overlapping observations

This is the step that turns redundancy into resolution. After regridding, the same
`(latitude, longitude)` node can appear more than once - once per orbital pass that covered
it. Each of those entries came from an independently fitted spectrum, so averaging them
gives a better estimate than keeping any single pass, and the improvement is largest exactly
where coverage is densest.

The strings written above are split back into numbers, grouped, averaged, and reassembled.

In [ ]:
# Operate on the formatted table produced above (or on the concatenated one).
df=new_df1

# Split the packed strings back into numeric pairs.
ratio_columns=[col for col in df.columns if '/Si_ratio' in col]

# One numeric column for the value, one for its uncertainty.
for col in ratio_columns:
    # DataFrame.applymap was renamed to DataFrame.map in pandas 2.1 and removed in 3.0.
    # The two are equivalent; `.map` is used here so the notebook runs on current pandas.
    df[[f"{col}_mean", f"{col}_uncertainty"]] = df[col].str.split("±", expand=True).map(str.strip)
    df[f"{col}_mean"] = pd.to_numeric(df[f"{col}_mean"], errors='coerce')
    df[f"{col}_uncertainty"] = pd.to_numeric(df[f"{col}_uncertainty"], errors='coerce')

# The packed columns are no longer needed.
df.drop(columns=ratio_columns, inplace=True)

# Collapse duplicate grid nodes by averaging every numeric column.
mean_columns = [col for col in df.columns if '_mean' in col]
uncertainty_columns = [col for col in df.columns if '_uncertainty' in col]


grouped_df = (
    df.groupby(['latitude', 'longitude'], as_index=False)
    .agg({**{col: 'mean' for col in mean_columns},
          **{col: 'mean' for col in uncertainty_columns}})
)

# Reassemble the display format.
for col in mean_columns:
    original_col = col.replace("_mean", "")
    grouped_df[original_col] = (
        grouped_df[col].round(5).astype(str) + " ± " +
        grouped_df[col.replace("_mean", "_uncertainty")].round(5).astype(str)
    )

# Discard the scratch columns.
grouped_df.drop(columns=mean_columns + uncertainty_columns, inplace=True)
grouped_df.head()

## Writing the grid

The resulting file is the only input notebook 04 needs.

In [ ]:
os.makedirs(os.path.dirname(GRID_CSV), exist_ok=True)
grouped_df.to_csv(GRID_CSV, index=False)
print(f"Grid written to {GRID_CSV} ({len(grouped_df)} cells)")